Every `for` loop in Python rests on a two-method protocol. Understanding that protocol — and the generator syntax built on top of it — unlocks a whole class of patterns: infinite sequences, lazy data pipelines, memory-efficient processing of huge datasets, and coroutines. This notebook starts at the protocol itself, builds up through custom iterators, introduces generator functions, covers `yield from` and generator delegation, shows how to send values into a generator, and ends with practical pipeline patterns.

## The Iterator Protocol

Python's `for` loop is syntactic sugar for two calls:

```python
for item in iterable:
    body
# is equivalent to:
iterator = iter(iterable)   # calls iterable.__iter__()
while True:
    try:
        item = next(iterator)  # calls iterator.__next__()
        body
    except StopIteration:
        break
```

Two roles:
- **Iterable**: any object with `__iter__()` — lists, strings, dicts, files, ranges, … Calling `iter()` on it returns an iterator.
- **Iterator**: any object with `__next__()` (and `__iter__` returning itself). Calling `next()` returns the next value or raises `StopIteration` when exhausted.

Iterators are **single-use** — once exhausted, they are done. Iterables can produce a fresh iterator each time `iter()` is called.

In [ ]:
# Manually drive the protocol
numbers = [10, 20, 30]

it = iter(numbers)          # get an iterator from the list
print(type(it))             # <class 'list_iterator'>

print(next(it))             # 10
print(next(it))             # 20
print(next(it))             # 30

try:
    print(next(it))         # raises StopIteration
except StopIteration:
    print("Exhausted!")

# Lists are iterable but NOT iterators
# Calling iter() on the same list gives a fresh iterator each time
it2 = iter(numbers)
print(next(it2))            # 10 — starts from the beginning again

In [ ]:
# Iterators are their own iterators — iter(iterator) returns self
it = iter([1, 2, 3])
print(iter(it) is it)       # True

# This means iterators work in for loops too
it = iter(["a", "b", "c"])
next(it)                    # consume 'a'
for item in it:             # resumes from 'b'
    print(item)             # b, c

In [ ]:
# next() with a default — avoids StopIteration when you just want one item
it = iter([42])
print(next(it, None))       # 42
print(next(it, None))       # None — exhausted, no exception
print(next(it, "done"))     # done

## Custom Iterators

Any class that implements `__iter__` and `__next__` becomes an iterator. This is the class-based approach — useful when you need mutable state, lazy computation, or precise control over the traversal.

In [ ]:
class CountUp:
    """Iterates integers from start up to (not including) stop."""

    def __init__(self, start, stop):
        self.current = start
        self.stop    = stop

    def __iter__(self):
        return self        # the object is its own iterator

    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration
        value = self.current
        self.current += 1
        return value


for n in CountUp(3, 8):
    print(n, end=" ")   # 3 4 5 6 7
print()

# Works with all iterator consumers
print(list(CountUp(1, 6)))       # [1, 2, 3, 4, 5]
print(sum(CountUp(1, 11)))       # 55
print(max(CountUp(5, 10)))       # 9

In [ ]:
# Separating the iterable from the iterator
# The iterable can be reused; the iterator is single-use

class FibSequence:
    """Iterable that produces Fibonacci numbers up to a limit."""

    def __init__(self, limit):
        self.limit = limit

    def __iter__(self):
        # Each call to __iter__ returns a *new* iterator
        return FibIterator(self.limit)


class FibIterator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.a > self.limit:
            raise StopIteration
        value = self.a
        self.a, self.b = self.b, self.a + self.b
        return value


fibs = FibSequence(100)

# Can iterate multiple times — each gives a fresh iterator
print(list(fibs))    # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]
print(list(fibs))    # same sequence — fresh iterator

## Generator Functions

Writing a full iterator class is verbose. **Generator functions** produce iterators with a fraction of the code. Any function that uses `yield` is a generator function. Calling it returns a **generator object** — an iterator — without executing the body yet.

Execution runs until a `yield`, pauses (saving the entire local state), and resumes from that exact point on the next `next()` call. When the function returns (or falls off the end), `StopIteration` is raised automatically.

In [ ]:
def count_up(start, stop):
    """Generator version of CountUp — same logic, far less code."""
    current = start
    while current < stop:
        yield current
        current += 1


gen = count_up(3, 8)
print(type(gen))            # <class 'generator'>
print(next(gen))            # 3
print(next(gen))            # 4

for n in count_up(3, 8):
    print(n, end=" ")       # 3 4 5 6 7
print()

print(list(count_up(1, 6))) # [1, 2, 3, 4, 5]

In [ ]:
# Generators are lazy — they compute items on demand
def verbose_range(n):
    print(f"  Starting generator (n={n})")
    for i in range(n):
        print(f"  Yielding {i}")
        yield i
        print(f"  Resumed after yield {i}")
    print(f"  Generator done")

gen = verbose_range(3)
print("Generator created — nothing has run yet")
print()

v = next(gen)
print(f"Got: {v}")
print()

v = next(gen)
print(f"Got: {v}")

In [ ]:
# Fibonacci as a generator — compare to the class-based version above
def fibonacci(limit=None):
    a, b = 0, 1
    while limit is None or a <= limit:
        yield a
        a, b = b, a + b


print(list(fibonacci(100)))   # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]

# Take first 10 from the infinite variant
import itertools
first10 = list(itertools.islice(fibonacci(), 10))
print(first10)   # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

## Infinite Generators

A generator function can loop forever — it is safe because values are only produced when asked for. Combined with `itertools.islice`, `itertools.takewhile`, or a manual `break`, you can consume exactly as many values as you need.

In [ ]:
def integers_from(n=0):
    """Yield integers starting from n, forever."""
    while True:
        yield n
        n += 1


def primes():
    """Yield prime numbers indefinitely."""
    found = []
    for n in integers_from(2):
        if all(n % p != 0 for p in found):
            found.append(n)
            yield n


# Take the first 15 primes
print(list(itertools.islice(primes(), 15)))

# Take primes below 50
print(list(itertools.takewhile(lambda p: p < 50, primes())))

# Cycle through a list forever — built-in infinite iterator
colours = itertools.cycle(["red", "green", "blue"])
print([next(colours) for _ in range(7)])

## `yield from` — Generator Delegation

`yield from iterable` delegates to another iterable, yielding all its values in turn. It is cleaner than a `for` loop with individual `yield` calls and also properly connects `send()` and `throw()` calls through the delegation chain.

In [ ]:
def flatten(nested):
    """Recursively flatten an arbitrarily nested iterable."""
    for item in nested:
        if isinstance(item, (list, tuple)):
            yield from flatten(item)   # delegate recursively
        else:
            yield item


data = [1, [2, 3], [4, [5, 6]], [[7, [8, 9]]]]
print(list(flatten(data)))    # [1, 2, 3, 4, 5, 6, 7, 8, 9]

In [ ]:
# Chain multiple generators with yield from
def chain_generators(*iterables):
    for it in iterables:
        yield from it


result = list(chain_generators([1, 2], "abc", range(5, 8)))
print(result)    # [1, 2, 'a', 'b', 'c', 5, 6, 7]

# yield from also captures the return value of a sub-generator
def sub():
    yield 1
    yield 2
    return "sub done"   # return value of a generator

def outer():
    result = yield from sub()   # result receives the sub-generator's return value
    print(f"Sub returned: {result!r}")
    yield 3

print(list(outer()))    # Sub returned: 'sub done'  →  [1, 2, 3]

## Sending Values into a Generator

`yield` is an expression, not just a statement. The `.send(value)` method resumes the generator and makes `value` the result of the `yield` expression inside it. This turns a generator into a coroutine — a function that can both produce and receive values.

The first call to start a generator must use `next(gen)` or `gen.send(None)` (priming it to the first `yield`). After that, any value can be sent.

In [ ]:
def running_average():
    """Coroutine that maintains a running average.
    Send numbers in; yields the current average after each.
    """
    total = 0
    count = 0
    average = None
    while True:
        value = yield average    # pause, send average out, receive next value
        if value is None:
            break
        total += value
        count += 1
        average = total / count


avg = running_average()
next(avg)                   # prime — advance to first yield

for n in [10, 20, 30, 40, 50]:
    result = avg.send(n)
    print(f"Sent {n:3}, running average: {result:.1f}")

In [ ]:
# .throw() — inject an exception into the generator
# .close() — GeneratorExit is thrown, causing the generator to clean up and stop

def guarded_gen():
    try:
        while True:
            value = yield
            print(f"  Processing: {value}")
    except ValueError as e:
        print(f"  Caught ValueError: {e}")
        yield "recovered"
    finally:
        print("  Generator cleanup")


g = guarded_gen()
next(g)
g.send(1)
g.send(2)
result = g.throw(ValueError, "bad value")
print(f"After throw: {result!r}")
g.close()

## Generator Return Values

A generator function can have a `return` statement with a value. The value is not yielded — it is stored as the `value` attribute of the `StopIteration` exception. `yield from` captures it automatically. In regular iteration the return value is discarded.

In [ ]:
def counted_sum(iterable):
    """Yield each item, then return (total, count) when done."""
    total = 0
    count = 0
    for item in iterable:
        total += item
        count += 1
        yield item
    return total, count    # stored in StopIteration.value


gen = counted_sum([10, 20, 30, 40])

try:
    while True:
        print(next(gen), end=" ")
except StopIteration as e:
    total, count = e.value
    print(f"\nTotal={total}, Count={count}")

## `itertools` — The Iterator Toolkit

The `itertools` module provides a set of fast, memory-efficient building blocks for iterator-based code. They all return lazy iterators — nothing is computed until consumed.

In [ ]:
import itertools

# ─── Infinite iterators ───────────────────────────────────────────────────
# count(start, step) — count from start
print(list(itertools.islice(itertools.count(10, 5), 5)))   # [10, 15, 20, 25, 30]

# cycle(iterable) — repeat forever
print(list(itertools.islice(itertools.cycle("ABC"), 8)))   # ['A','B','C','A','B','C','A','B']

# repeat(value, n) — repeat value n times (infinite if n omitted)
print(list(itertools.repeat(0, 5)))   # [0, 0, 0, 0, 0]

In [ ]:
# ─── Terminating iterators ────────────────────────────────────────────────
# chain — concatenate iterables
print(list(itertools.chain([1,2], [3,4], [5])))   # [1, 2, 3, 4, 5]

# compress — filter by a boolean mask
data  = ["A", "B", "C", "D", "E"]
mask  = [1, 0, 1, 0, 1]
print(list(itertools.compress(data, mask)))        # ['A', 'C', 'E']

# dropwhile / takewhile — conditional slicing
nums = [1, 3, 5, 2, 4, 6, 7]
print(list(itertools.dropwhile(lambda n: n < 4, nums)))   # [5, 2, 4, 6, 7] — drops while < 4
print(list(itertools.takewhile(lambda n: n < 6, nums)))   # [1, 3, 5, 2, 4] — takes while < 6

# pairwise (Python 3.10+) — overlapping pairs
print(list(itertools.pairwise([1, 2, 3, 4, 5])))  # [(1,2),(2,3),(3,4),(4,5)]

In [ ]:
# ─── Combinatoric iterators ───────────────────────────────────────────────
# product — Cartesian product
print(list(itertools.product("AB", [1, 2])))       # [('A',1),('A',2),('B',1),('B',2)]

# permutations — ordered arrangements
print(list(itertools.permutations("ABC", 2)))      # all 2-item ordered pairs

# combinations — unordered selections (no repeats)
print(list(itertools.combinations("ABCD", 2)))     # all 2-item unordered pairs

# combinations_with_replacement — same but item can repeat
print(list(itertools.combinations_with_replacement("AB", 2)))  # [('A','A'),('A','B'),('B','B')]

## Lazy Pipelines

Generators compose naturally into **pipelines**: each stage is a generator that consumes from the previous stage and yields to the next. The entire pipeline runs lazily — data flows through one item at a time, with no intermediate collections stored in memory.

In [ ]:
# A generator pipeline for processing log lines
import re
from datetime import datetime

# Stage 1: produce lines (could be a real file; use a list here)
def lines_source():
    raw = [
        "2024-01-15 09:03:12 ERROR  db timeout after 30s",
        "2024-01-15 09:03:14 INFO   reconnecting",
        "2024-01-15 09:03:15 ERROR  query failed: table not found",
        "2024-01-15 09:03:16 DEBUG  pool size: 10",
        "2024-01-15 09:04:01 ERROR  connection refused",
        "2024-01-15 09:04:05 WARNING high memory usage",
    ]
    yield from raw


# Stage 2: parse each line into a dict
LOG_RE = re.compile(r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) (\w+)\s+(.+)")

def parse_lines(lines):
    for line in lines:
        if m := LOG_RE.match(line):
            yield {
                "ts":    m.group(1),
                "level": m.group(2),
                "msg":   m.group(3).strip(),
            }


# Stage 3: filter to ERROR level only
def errors_only(records):
    for rec in records:
        if rec["level"] == "ERROR":
            yield rec


# Stage 4: format for display
def format_records(records):
    for rec in records:
        yield f"[{rec['ts']}] {rec['msg']}"


# Wire up the pipeline — nothing runs until we iterate the final stage
pipeline = format_records(errors_only(parse_lines(lines_source())))

for line in pipeline:
    print(line)

In [ ]:
# Memory comparison: list pipeline vs generator pipeline
import sys

N = 100_000

# List pipeline — each stage allocates a full list
numbers_list = list(range(N))
evens_list   = [n for n in numbers_list if n % 2 == 0]
squares_list = [n ** 2 for n in evens_list]
result_list  = sum(squares_list)

print(f"List approach:      {sys.getsizeof(numbers_list) + sys.getsizeof(evens_list) + sys.getsizeof(squares_list):>12,} bytes")

# Generator pipeline — no intermediate lists
numbers_gen = range(N)
evens_gen   = (n for n in numbers_gen if n % 2 == 0)
squares_gen = (n ** 2 for n in evens_gen)
result_gen  = sum(squares_gen)

print(f"Generator approach: {sys.getsizeof(evens_gen) + sys.getsizeof(squares_gen):>12,} bytes")
print(f"Same result: {result_list == result_gen}")

## Summary

- The **iterator protocol**: `__iter__()` returns an iterator; `__next__()` returns the next value or raises `StopIteration`. `iter(x)` and `next(x)` call these methods. Iterables can produce fresh iterators repeatedly; iterators are single-use.
- `next(iterator, default)` avoids `StopIteration` by returning a default when exhausted.
- **Custom iterators** implement `__iter__` and `__next__`. Separate the iterable (which produces a fresh iterator on each `__iter__` call) from the iterator (which holds the traversal state).
- **Generator functions** use `yield` to pause and resume. Calling the function returns a generator object (lazy iterator). The body does not run until the first `next()`. State is fully preserved across yields.
- **Infinite generators** are safe — produce values only when consumed. Use `itertools.islice` or `itertools.takewhile` to limit consumption.
- **`yield from`** delegates to another iterable, transparently forwarding `send()` and `throw()`. It also captures the sub-generator's return value.
- **`.send(value)`** resumes a generator and makes `value` the result of the paused `yield` expression — turning the generator into a coroutine. The first call must be `next()` or `.send(None)` to prime it.
- **Generator return values** are stored in `StopIteration.value`; `yield from` captures them automatically.
- **`itertools`** provides infinite iterators (`count`, `cycle`, `repeat`), terminating combiners (`chain`, `compress`, `dropwhile`, `takewhile`, `pairwise`), and combinatorics (`product`, `permutations`, `combinations`).
- **Lazy pipelines** chain generator functions so data flows one item at a time with no intermediate collections — constant memory regardless of input size.